In [11]:
!pip install pandas tqdm tree_sitter


In [12]:
import requests

# رابط الملف من Hugging Face
url = "https://huggingface.co/datasets/LorenzH/juliet_test_suite_c_1_3/resolve/main/jts_c_1_3_train.csv?download=true"
filename = "jts_c_1_3_train.csv"

print("جاري تحميل الملف... قد يستغرق ذلك وقتاً حسب حجم الملف.")

# بدء عملية التحميل
response = requests.get(url, stream=True)

if response.status_code == 200:
    with open(filename, 'wb') as f:
        f.write(response.content)
    print(f"✅ تم بنجاح! الملف متاح الآن باسم: {filename}")
else:
    print(f"❌ فشل التحميل. كود الخطأ: {response.status_code}")

جاري تحميل الملف... قد يستغرق ذلك وقتاً حسب حجم الملف.
✅ تم بنجاح! الملف متاح الآن باسم: jts_c_1_3_train.csv


In [13]:
import pandas as pd
import json
import re
from tqdm import tqdm

# ---------- Paths ----------
juliet_csv = "/kaggle/working/jts_c_1_3_train.csv"
output_path = "stage1_juliet_normalized.jsonl"

# ---------- Helpers ----------
def clean_code(code):
    code = str(code)
    code = re.sub(r"//.*", "", code)
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

# ---------- Load ----------
print("📥 Loading Juliet CSV...")
df = pd.read_csv(juliet_csv, low_memory=False)

print("\n🔍 Detected columns:")
print(list(df.columns))
print(f"\n📦 Total rows: {len(df)}")

records = []
func_id = 0

# ---------- Process ----------
for idx, row in tqdm(df.iterrows(), total=len(df), desc="🛠 Processing Juliet"):
    # BAD = vulnerable
    bad_code = row.get("bad", "")
    if isinstance(bad_code, str) and bad_code.strip():
        records.append({
            "function_id": f"juliet_bad_{func_id}",
            "code": clean_code(bad_code),
            "label": "VULN",
            "dataset": "Juliet",
            "cwe": row.get("class", "UNKNOWN")
        })
        func_id += 1

    # GOOD = safe
    good_code = row.get("good", "")
    if isinstance(good_code, str) and good_code.strip():
        records.append({
            "function_id": f"juliet_good_{func_id}",
            "code": clean_code(good_code),
            "label": "SAFE",
            "dataset": "Juliet",
            "cwe": row.get("class", "UNKNOWN")
        })
        func_id += 1

print(f"\n✅ Total extracted functions: {len(records)}")

# ---------- Save ----------
print(f"\n💾 Saving Stage 1 Juliet → {output_path}")
with open(output_path, "w") as f:
    for rec in tqdm(records, desc="📦 Writing JSONL"):
        f.write(json.dumps(rec) + "\n")

print("\n🎉 Stage 1 (Juliet) COMPLETED SUCCESSFULLY!")
print(f"📄 Output file: {output_path}")


📥 Loading Juliet CSV...

🔍 Detected columns:
['index', 'filename', 'class', 'good', 'bad']

📦 Total rows: 80706


🛠 Processing Juliet: 100%|██████████| 80706/80706 [00:10<00:00, 7894.76it/s]



✅ Total extracted functions: 161412

💾 Saving Stage 1 Juliet → stage1_juliet_normalized.jsonl


📦 Writing JSONL: 100%|██████████| 161412/161412 [00:00<00:00, 188702.35it/s]



🎉 Stage 1 (Juliet) COMPLETED SUCCESSFULLY!
📄 Output file: stage1_juliet_normalized.jsonl


In [ ]:
import pandas as pd
import json
import re
from collections import Counter
from tqdm import tqdm

# ---------- Paths ----------
msr_path = "/kaggle/input/msr-data/MSR_data_cleaned.csv"
juliet_path = "/kaggle/working/stage1_juliet_normalized.jsonl"
megavul_path = "/kaggle/input/megavul-a-cc-java-vulnerability-dataset/megavul/c_cpp/megavul.json"

output_path = "stage1_all_normalized.jsonl"

# ---------- Helpers ----------
def clean_code(code):
    if not code:
        return ""
    code = re.sub(r"//.*", "", code)
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"\s+", " ", code).strip()
    return code

def detect_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

# ---------- MSR ----------
def process_msr(path):
    df = pd.read_csv(path, low_memory=False)
    print("\n🔍 MSR columns detected:")
    print(list(df.columns))

    code_col = detect_column(df, ["func_before", "code", "Code"])
    label_col = detect_column(df, ["vul"])
    cwe_col = detect_column(df, ["cwe", "CWE", "cwe_id", "vulnerability_type"])

    if code_col is None or label_col is None:
        raise RuntimeError("❌ MSR ERROR: required columns not found")

    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="MSR"):
        code = clean_code(str(row[code_col]))
        if not code:
            continue

        label = "VULN" if int(row[label_col]) == 1 else "SAFE"
        cwe = str(row[cwe_col]).strip() if cwe_col and pd.notna(row.get(cwe_col)) else "UNKNOWN"

        records.append({
            "function_id": f"msr_{idx}",
            "code": code,
            "label": label,
            "dataset": "MSR",
            "cwe": cwe
        })

    print(f"✅ MSR processed: {len(records)}")
    return records

# ---------- Juliet (READY – preserves CWE) ----------
def process_juliet(path):
    records = []
    with open(path, "r") as f:
        for line in tqdm(f, desc="Juliet"):
            obj = json.loads(line)
            records.append({
                "function_id": obj["function_id"],
                "code": clean_code(obj["code"]),
                "label": obj["label"],
                "dataset": "Juliet",
                "cwe": obj.get("cwe", "UNKNOWN")
            })
    print(f"✅ Juliet processed: {len(records)}")
    return records

# ---------- MegaVul ----------
def process_megavul(path):
    records = []
    with open(path, "r") as f:
        data = json.load(f)

    for idx, item in tqdm(enumerate(data), total=len(data), desc="MegaVul"):
        code = clean_code(item.get("code", ""))
        if not code:
            continue

        label = "VULN" if int(item["label"]) == 1 else "SAFE"
        cwe = item.get("cwe", item.get("CWE", item.get("cwe_id", "UNKNOWN")))

        records.append({
            "function_id": f"megavul_{idx}",
            "code": code,
            "label": label,
            "dataset": "MegaVul",
            "cwe": str(cwe) if cwe else "UNKNOWN"
        })

    print(f"✅ MegaVul processed: {len(records)}")
    return records

# ---------- Main ----------
all_records = []
all_records += process_msr(msr_path)
all_records += process_juliet(juliet_path)
all_records += process_megavul(megavul_path)

# ---------- Save ----------
print("\n💾 Saving Stage 1 unified JSONL...")
with open(output_path, "w") as f:
    for rec in tqdm(all_records, desc="📦 Writing"):
        f.write(json.dumps(rec) + "\n")

print(f"\n🎉 Stage 1 DONE | Total functions: {len(all_records)}")

# ---------- Stats ----------
dataset_counts = Counter(r["dataset"] for r in all_records)
label_counts = Counter(r["label"] for r in all_records)
cwe_counts = Counter(r["cwe"] for r in all_records)

print("\n=== Dataset distribution ===")
for k, v in dataset_counts.items():
    print(k, v)

print("\n=== Label distribution ===")
for k, v in label_counts.items():
    print(k, v)

print(f"\n=== CWE types found: {len(cwe_counts)} ===")
for k, v in cwe_counts.most_common(15):
    print(f"  {k}: {v}")



🔍 MSR columns detected:
['Unnamed: 0', 'Access Gained', 'Attack Origin', 'Authentication Required', 'Availability', 'CVE ID', 'CVE Page', 'CWE ID', 'Complexity', 'Confidentiality', 'Integrity', 'Known Exploits', 'Publish Date', 'Score', 'Summary', 'Update Date', 'Vulnerability Classification', 'add_lines', 'codeLink', 'commit_id', 'commit_message', 'del_lines', 'file_name', 'files_changed', 'func_after', 'func_before', 'lang', 'lines_after', 'lines_before', 'parentID', 'patch', 'project', 'project_after', 'project_before', 'vul', 'vul_func_with_fix']


MSR: 100%|██████████| 188636/188636 [00:14<00:00, 13229.20it/s]


✅ MSR processed: 188636


Juliet: 161412it [00:04, 33786.88it/s]


✅ Juliet processed: 161412


MegaVul: 100%|██████████| 353873/353873 [00:00<00:00, 1720064.97it/s]


✅ MegaVul processed: 0

💾 Saving Stage 1 unified JSONL...


📦 Writing: 100%|██████████| 350048/350048 [00:02<00:00, 169442.24it/s]



🎉 Stage 1 DONE | Total functions: 350048

=== Dataset distribution ===
MSR 188636
Juliet 161412

=== Label distribution ===
SAFE 258442
VULN 91606


In [ ]:
import json
import re
from tqdm import tqdm

# ---------- Paths ----------
stage1_path = "/kaggle/working/stage1_all_normalized.jsonl"
stage2_output = "stage2_features.jsonl"

# ---------- Dangerous APIs ----------
DANGEROUS_APIS = [
    "strcpy", "strcat", "sprintf", "gets", "scanf",
    "memcpy", "memmove", "malloc", "free"
]

# ---------- Helper functions ----------
def clean_code(code):
    if not code:
        return ""
    code = re.sub(r"//.*", "", code)
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"\s+", " ", code).strip()
    return code

def extract_dangerous_api_features(code):
    return {f"api_{api}": code.count(api) for api in DANGEROUS_APIS}

def extract_memory_features(code):
    return {
        "uses_malloc": int("malloc" in code),
        "uses_free": int("free" in code),
        "malloc_count": code.count("malloc"),
        "free_count": code.count("free"),
    }

def extract_pointer_features(code):
    return {
        "ptr_star": code.count("*"),
        "ptr_amp": code.count("&"),
        "ptr_arrow": code.count("->"),
    }

def extract_code_stats(code):
    return {
        "code_length": len(code),
        "num_statements": code.count(";"),
        "num_lines": code.count("\n") + 1
    }

# ---------- Load Stage 1 ----------
print(f"\n📥 Loading Stage 1 from: {stage1_path}")

output_records = []
skipped = 0

with open(stage1_path, "r") as f:
    for line in tqdm(f, desc="🛠 Extracting Stage 2 Features"):
        try:
            obj = json.loads(line)
            raw_code = obj.get("code", "")
            code = clean_code(raw_code)

            # ---------- Features ----------
            if not code:
                features = {f"api_{api}": 0 for api in DANGEROUS_APIS}
                features.update({
                    "uses_malloc": 0,
                    "uses_free": 0,
                    "malloc_count": 0,
                    "free_count": 0,
                    "ptr_star": 0,
                    "ptr_amp": 0,
                    "ptr_arrow": 0,
                    "code_length": 0,
                    "num_statements": 0,
                    "num_lines": 0
                })
            else:
                features = {}
                features.update(extract_dangerous_api_features(code))
                features.update(extract_memory_features(code))
                features.update(extract_pointer_features(code))
                features.update(extract_code_stats(code))

            # ---------- IMPORTANT: Preserve CWE ----------
            record = {
                "function_id": obj["function_id"],
                "code": code,
                "dataset": obj["dataset"],
                "label": obj["label"],
                "cwe": obj.get("cwe", "UNKNOWN"),
                "features": features
            }

            output_records.append(record)

        except Exception:
            skipped += 1

# ---------- Save Output ----------
print(f"\n💾 Saving Stage 2 output → {stage2_output}")
with open(stage2_output, "w") as f:
    for rec in output_records:
        f.write(json.dumps(rec) + "\n")

# ---------- Summary ----------
print("\n🎉 Stage 2 COMPLETED SUCCESSFULLY")
print(f"✅ Processed functions : {len(output_records)}")
print(f"⚠️ Skipped functions   : {skipped}")
print(f"📄 Output file         : {stage2_output}")



📥 Loading Stage 1 from: /kaggle/working/stage1_all_normalized.jsonl


🛠 Extracting Stage 2 Features: 350048it [00:17, 19992.98it/s]



💾 Saving Stage 2 output → stage2_features.jsonl

🎉 Stage 2 COMPLETED SUCCESSFULLY
✅ Processed functions : 350048
⚠️ Skipped functions   : 0
📄 Output file         : stage2_features.jsonl


In [16]:
import json

count = 0
with open("/kaggle/working/stage2_features.jsonl") as f:
    for line in f:
        obj = json.loads(line)
        count += 1
        if count < 5:
            print(obj)
print("Total lines:", count)


{'function_id': 'msr_0', 'code': 'static bool check_rodc_critical_attribute(struct ldb_message *msg) { uint32_t schemaFlagsEx, searchFlags, rodc_filtered_flags; schemaFlagsEx = ldb_msg_find_attr_as_uint(msg, "schemaFlagsEx", 0); searchFlags = ldb_msg_find_attr_as_uint(msg, "searchFlags", 0); rodc_filtered_flags = (SEARCH_FLAG_RODC_ATTRIBUTE | SEARCH_FLAG_CONFIDENTIAL); if ((schemaFlagsEx & SCHEMA_FLAG_ATTR_IS_CRITICAL) && ((searchFlags & rodc_filtered_flags) == rodc_filtered_flags)) { return true; } else { return false; } }', 'dataset': 'MSR', 'label': 'SAFE', 'features': {'api_strcpy': 0, 'api_strcat': 0, 'api_sprintf': 0, 'api_gets': 0, 'api_scanf': 0, 'api_memcpy': 0, 'api_memmove': 0, 'api_malloc': 0, 'api_free': 0, 'uses_malloc': 0, 'uses_free': 0, 'malloc_count': 0, 'free_count': 0, 'ptr_star': 1, 'ptr_amp': 4, 'ptr_arrow': 0, 'code_length': 495, 'num_statements': 6, 'num_lines': 1}}
{'function_id': 'msr_1', 'code': 'static int samldb_add_entry(struct samldb_ctx *ac) { struct ldb

In [17]:
import json

with open("stage2_features.jsonl") as f:
    sample = json.loads(next(f))
    print(sample.keys())



dict_keys(['function_id', 'code', 'dataset', 'label', 'features'])


In [ ]:
# =========================
# Stage 3 – Multi-Class CWE Classification
# Hybrid CodeBERT + Static Features
# With Attention for Explainability
# FP16 + EarlyStopping + Resume
# =========================

import json, os, gc, pickle
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from torch.optim import AdamW
from tqdm import tqdm
from collections import Counter

# ---------- Paths ----------
STAGE2_PATH = "/kaggle/working/stage2_features.jsonl"
CHECKPOINT_PATH = "stage3_checkpoint.pt"
BEST_MODEL_PATH = "stage3_best.pt"
CWE_MAP_PATH = "cwe_to_id.json"
SCALER_PATH = "scaler.pkl"

# ---------- Hyperparameters ----------
BATCH_SIZE = 8
EPOCHS = 12
FREEZE_EPOCHS = 2
LR = 2e-5
MAX_LEN = 256
PATIENCE = 3          # early stopping
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# Load Stage 2 & Build CWE Map
# =========================
print("🔹 Loading Stage 2...")
raw_data = []
with open(STAGE2_PATH, "r") as f:
    for line in f:
        o = json.loads(line)
        raw_data.append(o)

print(f"📦 Total samples: {len(raw_data)}")

# ---------- Build CWE mapping ----------
# SAFE samples → CWE-0 (No Vulnerability)
# VULN samples → keep their CWE
cwe_set = set()
for o in raw_data:
    if o["label"] == "SAFE":
        cwe_set.add("CWE-0")
    else:
        cwe_raw = o.get("cwe", "UNKNOWN")
        cwe_str = str(cwe_raw).strip()
        if not cwe_str or cwe_str == "UNKNOWN":
            cwe_str = "CWE-UNKNOWN"
        elif not cwe_str.startswith("CWE-"):
            cwe_str = f"CWE-{cwe_str}"
        cwe_set.add(cwe_str)

# Sort for reproducibility (CWE-0 first)
sorted_cwes = sorted(cwe_set, key=lambda x: (x != "CWE-0", x))
cwe_to_id = {cwe: idx for idx, cwe in enumerate(sorted_cwes)}
id_to_cwe = {idx: cwe for cwe, idx in cwe_to_id.items()}
NUM_CLASSES = len(cwe_to_id)

print(f"🏷️ Number of CWE classes: {NUM_CLASSES}")
for cwe, cid in list(cwe_to_id.items())[:10]:
    print(f"  {cwe} → {cid}")
if NUM_CLASSES > 10:
    print(f"  ... and {NUM_CLASSES - 10} more")

# Save CWE mapping
with open(CWE_MAP_PATH, "w") as f:
    json.dump({"cwe_to_id": cwe_to_id, "id_to_cwe": id_to_cwe}, f, indent=2)
print(f"💾 CWE mapping saved → {CWE_MAP_PATH}")

# ---------- Prepare data ----------
data = []
for o in raw_data:
    if o["label"] == "SAFE":
        cwe_label = "CWE-0"
    else:
        cwe_raw = str(o.get("cwe", "UNKNOWN")).strip()
        if not cwe_raw or cwe_raw == "UNKNOWN":
            cwe_label = "CWE-UNKNOWN"
        elif not cwe_raw.startswith("CWE-"):
            cwe_label = f"CWE-{cwe_raw}"
        else:
            cwe_label = cwe_raw

    data.append({
        "code": o["code"],
        "features": list(o["features"].values()),
        "label": cwe_to_id[cwe_label],
        "cwe": cwe_label
    })

del raw_data
gc.collect()

# CWE distribution
cwe_dist = Counter(d["cwe"] for d in data)
print("\n📊 CWE Distribution (top 15):")
for cwe, cnt in cwe_dist.most_common(15):
    print(f"  {cwe}: {cnt}")

# =========================
# Feature Scaling
# =========================
X = np.array([d["features"] for d in data], dtype=np.float32)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Save scaler for inference
with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler, f)
print(f"💾 Scaler saved → {SCALER_PATH}")

for i in range(len(data)):
    data[i]["features"] = X[i]

FEAT_DIM = len(data[0]["features"])

# =========================
# Dataset (Lazy Tokenization)
# =========================
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

class HybridDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]
        enc = tokenizer(
            d["code"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        feat = torch.tensor(d["features"], dtype=torch.float32)
        label = torch.tensor(d["label"], dtype=torch.long)
        return enc, feat, label

dataset = HybridDataset(data)

# =========================
# Train / Val / Test Split
# =========================
n_total = len(dataset)
n_val = int(0.15 * n_total)
n_test = int(0.15 * n_total)
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# =========================
# Hybrid Model (Multi-Class + Attention)
# =========================
class HybridModel(nn.Module):
    def __init__(self, feat_dim, num_classes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("microsoft/codebert-base")
        self.feat_proj = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.cls = nn.Sequential(
            nn.Linear(768 + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, ids, mask, feat, return_attentions=False):
        outputs = self.encoder(
            input_ids=ids,
            attention_mask=mask,
            output_attentions=return_attentions
        )
        cls_emb = outputs.last_hidden_state[:, 0]  # [CLS] token
        f = self.feat_proj(feat)
        z = torch.cat([cls_emb, f], dim=1)
        logits = self.cls(z)

        if return_attentions:
            return logits, outputs.attentions
        return logits

model = HybridModel(FEAT_DIM, NUM_CLASSES).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
scaler_fp16 = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

# =========================
# Resume Training (if exists)
# =========================
start_epoch = 0
best_f1 = 0.0
patience_counter = 0

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    if ckpt.get("num_classes") == NUM_CLASSES:
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        best_f1 = ckpt["best_f1"]
        patience_counter = ckpt["patience"]
        print(f"🔁 Resuming from epoch {start_epoch}")
    else:
        print("⚠️ Checkpoint has different num_classes, starting fresh")

# =========================
# Freeze Encoder Initially
# =========================
for p in model.encoder.parameters():
    p.requires_grad = False

# =========================
# Evaluation
# =========================
def evaluate(loader):
    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for enc, feat, y in loader:
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            feat = feat.to(DEVICE)
            y = y.to(DEVICE)

            out = model(enc["input_ids"], enc["attention_mask"], feat)
            preds.extend(out.argmax(1).cpu().tolist())
            gold.extend(y.cpu().tolist())
    return f1_score(gold, preds, average="weighted")

# =========================
# Training Loop (CrossEntropyLoss for multi-class)
# =========================
for epoch in range(start_epoch, EPOCHS):

    if epoch == FREEZE_EPOCHS:
        for p in model.encoder.parameters():
            p.requires_grad = True
        print("🔓 Encoder Unfrozen")

    model.train()
    bar = tqdm(
        train_loader,
        desc=f"Epoch [{epoch+1}/{EPOCHS}]",
        leave=False
    )

    for enc, feat, y in bar:
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        feat = feat.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        if DEVICE == "cuda":
            with torch.amp.autocast("cuda"):
                out = model(enc["input_ids"], enc["attention_mask"], feat)
                loss = criterion(out, y)
            scaler_fp16.scale(loss).backward()
            scaler_fp16.step(optimizer)
            scaler_fp16.update()
        else:
            out = model(enc["input_ids"], enc["attention_mask"], feat)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        bar.set_postfix(loss=f"{loss.item():.4f}")

    val_f1 = evaluate(val_loader)
    print(f"📊 Epoch {epoch+1} | Val F1 (weighted): {val_f1:.4f}")

    # ---- Save checkpoint ----
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_f1": best_f1,
        "patience": patience_counter,
        "num_classes": NUM_CLASSES
    }, CHECKPOINT_PATH)

    # ---- Early stopping logic ----
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("💾 Best model saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("⏹ Early stopping triggered")
            break

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# =========================
# Final Test
# =========================
print("\n🧪 FINAL TEST")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
test_f1 = evaluate(test_loader)
print(f"🔥 Final Test F1 (weighted) = {test_f1:.4f}")
print(f"🏷️ Total CWE classes: {NUM_CLASSES}")


🔹 Loading Stage 2...
📦 Total samples: 350048


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

/tmp/ipykernel_55/3674466882.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_fp16 = torch.cuda.amp.GradScaler()


🔁 Resuming from epoch 10


Epoch [11/12]:   0%|          | 0/30630 [00:00<?, ?it/s]/tmp/ipykernel_55/3674466882.py:199: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


📊 Epoch 11 | Val F1: 0.9130
⏹ Early stopping triggered

🧪 FINAL TEST
🔥 Final Test F1 = 0.9219


In [ ]:
# =========================
# Stage 4 – Evaluation & Per-CWE Metrics
# =========================

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc,
    precision_recall_fscore_support
)

# =========================
# Evaluate on Test Set
# =========================
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

preds, gold, all_probs = [], [], []

with torch.no_grad():
    for enc, feat, y in test_loader:
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        feat = feat.to(DEVICE)
        y = y.to(DEVICE)
        out = model(enc["input_ids"], enc["attention_mask"], feat)
        prob = torch.softmax(out, dim=1)
        preds.extend(out.argmax(1).cpu().tolist())
        gold.extend(y.cpu().tolist())
        all_probs.extend(prob.cpu().numpy())

y_true = np.array(gold)
y_pred = np.array(preds)
y_probs = np.array(all_probs)

# =========================
# Classification Report
# =========================
present_ids = sorted(set(y_true.tolist() + y_pred.tolist()))
target_names = [id_to_cwe.get(i, f"class_{i}") for i in present_ids]

report = classification_report(
    y_true, y_pred,
    labels=present_ids,
    target_names=target_names,
    zero_division=0
)
print("\n📊 Multi-Class Classification Report:\n")
print(report)

f1_weighted = f1_score(y_true, y_pred, average="weighted")
f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"🔥 Test F1 (weighted): {f1_weighted:.4f}")
print(f"🔥 Test F1 (macro):    {f1_macro:.4f}")

# =========================
# ROC Curve (binary: SAFE vs any VULN)
# =========================
cwe0_id = cwe_to_id.get("CWE-0", 0)
y_true_bin = (y_true != cwe0_id).astype(int)
y_score_bin = 1.0 - y_probs[:, cwe0_id]

fpr, tpr, _ = roc_curve(y_true_bin, y_score_bin)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (SAFE vs VULNERABLE)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig('ROC_Curve.pdf')
plt.show()

# =========================
# Confusion Matrix (Top-10 CWE classes)
# =========================
support = {cid: int((y_true == cid).sum()) for cid in present_ids}
top_ids = sorted(present_ids, key=lambda x: support[x], reverse=True)[:10]
top_names = [id_to_cwe.get(i, f"cls_{i}") for i in top_ids]

mask_top = np.isin(y_true, top_ids)
cm = confusion_matrix(y_true[mask_top], y_pred[mask_top], labels=top_ids)

plt.figure(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top_names)
disp.plot(cmap=plt.cm.Blues, values_format='d', xticks_rotation=45)
plt.title(f'Confusion Matrix — Top 10 CWE Classes (F1: {f1_weighted:.4f})')
plt.tight_layout()
plt.savefig('Confusion_Matrix.pdf')
plt.show()

# =========================
# Top-5 Best & Worst CWE classes
# =========================
prec, rec, f1s, sup = precision_recall_fscore_support(
    y_true, y_pred, labels=present_ids, zero_division=0
)
cwe_scores = []
for i, cid in enumerate(present_ids):
    cwe_scores.append({
        "cwe": id_to_cwe.get(cid, f"class_{cid}"),
        "f1": f1s[i], "precision": prec[i],
        "recall": rec[i], "support": int(sup[i])
    })
cwe_scores.sort(key=lambda x: x["f1"], reverse=True)

print("\n🏆 Top-5 Best CWE classes:")
for s in cwe_scores[:5]:
    print(f"  {s['cwe']}: F1={s['f1']:.3f} P={s['precision']:.3f} R={s['recall']:.3f} (n={s['support']})")

print("\n⚠️ Top-5 Worst CWE classes:")
for s in cwe_scores[-5:]:
    print(f"  {s['cwe']}: F1={s['f1']:.3f} P={s['precision']:.3f} R={s['recall']:.3f} (n={s['support']})")

# =========================
# Summary
# =========================
print("\n" + "=" * 40)
print(f"✅ FINAL RESULTS")
print(f"  F1 (weighted): {f1_weighted:.4f}")
print(f"  F1 (macro):    {f1_macro:.4f}")
print(f"  ROC AUC:       {roc_auc:.4f}")
print(f"  CWE Classes:   {NUM_CLASSES}")
print("=" * 40)


🔹 Loading Stage 2...
📦 Total samples: 350048
📊 Label distribution: {'SAFE': 258442, 'VULN': 91606}

📊 Classification Report:
               precision    recall  f1-score   support

        SAFE       0.96      0.98      0.97     38718
        VULN       0.95      0.90      0.92     13789

    accuracy                           0.96     52507
   macro avg       0.96      0.94      0.95     52507
weighted avg       0.96      0.96      0.96     52507

📉 Confusion Matrix:
 [[38026   692]
 [ 1405 12384]]

🔥 SAFE Metrics:
  Precision: 0.9644
  Recall   : 0.9821
  F1       : 0.9732

🔥 VULN Metrics:
  Precision: 0.9471
  Recall   : 0.8981
  F1       : 0.9219

✅ Stage 4 COMPLETED
